# Objetivo
Generar samples y obtener una idea del escalado temporal, todo aprovechando de usar tecnicas de batching programadas; de esta manera generando datos para los modelos de ML clasificativos.

In [8]:
import numpy as np
import pandas as pd
import time
import os
#import psutil
import glob
import pickle
from scipy.stats import qmc
from lib.oracle import OracleExecutor  # assumes your OracleExecutor is in oracle_wrapper.py

epsilon = 0.001
vev = 246

def generate_local_variations(
    m_phi_base: float,
    m_A_center: float,
    m12_center: float,
    batch_size: int,
    eps_A: float,
    eps_m12: float,
    seed: int = None
    ) -> np.ndarray:
    """
    Generate `batch_size` points where only m_A and m12_2 vary in a small Latin 
    Hypercube around (m_A_center, m12_center), and all other 5 dims are fixed.

    Returns an array of shape (batch_size, 7) with column order:
      [m_phi, m_A, sin_ba, tan_beta, lambda6, lambda7, m12_2]
    """

    # 1) LatinHypercube in 2D (for m_A and m12)
    sampler = qmc.LatinHypercube(d=2)
    unit = sampler.random(n=batch_size)

    # 2) Scale to [m_A_center±eps_A] × [m12_center±eps_m12]
    bounds = np.array([
        [m_A_center - eps_A, m_A_center + eps_A],
        [m12_center - eps_m12, m12_center + eps_m12]
    ])
    scaled = qmc.scale(unit, bounds[:,0], bounds[:,1])  # shape (batch_size,2)

    # 3) Build the full parameter array, hard-coding the other 5 dims:
    P = np.empty((batch_size, 7), dtype=float)
    P[:, 0] = m_phi_base           # m_phi
    P[:, 1] = scaled[:, 0]         # m_A (varying)
    P[:, 2] = 1.0                  # sin(beta - alpha), fixed
    P[:, 3] = 10000.0              # tan(beta), fixed
    P[:, 4] = 0.1                  # lambda_6, fixed
    P[:, 5] = 0.0                  # lambda_7, fixed
    P[:, 6] = scaled[:, 1]         # m12_2 (varying)

    return P


def get_parameters_from_points(
    csv_path: str,
    batch_size: int,
    eps_A: float,
    eps_m12: float,
    seed: int = None
    ) -> np.ndarray:
    """
    Reads `csv_path` containing base points with columns 'Mh2', 'Mh3', 'm12_2' and
    for each row generates `batch_size` local variations via generate_local_variations.
    Returns a combined array of shape (n_rows * batch_size, 7).
    """
    df = pd.read_csv(csv_path)
    all_batches = []
    for idx, row in df.iterrows():
        m_phi_base   = row["Mh2"]    # heavy CP-even Higgs mass as m_phi
        m_A_center   = row["Mh3"]    # CP-odd Higgs mass as m_A
        m12_center   = row["m12_2"]
        # derive unique seed per batch for reproducibility
        batch_seed = None if seed is None else seed + idx
        P = generate_local_variations(
            m_phi_base, m_A_center, m12_center,
            batch_size, eps_A, eps_m12, seed=batch_seed
        )
        all_batches.append(P)
    # stack all batches into one array
    return np.vstack(all_batches)

# Prepare executor
executor = OracleExecutor(nthreads=4)



In [11]:
def multiple_runs(
    csv_path: str,
    N_repeat_runs: int,
    batch_size: int,
    eps_A: float,
    eps_m12: float,
    outdir: str,
    executor: OracleExecutor,
    base_seed: int = 42
):
    """
    Perform n_runs of local LH sampling around each of the first n_runs base points
    in `csv_path`.  For run j:
      - Read row j from CSV (must contain Mh2, Mh3, m12_2)
      - Generate `batch_size` variations in (m_A, m12_2)
      - Call executor.map on the parametrizations
      - Dump to outdir/batch_{idx}_{m_phi_base:.0f}.pkl
    
    Also prints a rough time estimate before running.
    """
    # Load your base points
    df = pd.read_csv(csv_path)
    n_runs = len(df)


    # Rough time estimate (seconds per point)
    time_per_point = 415.7 / 15_000
    total_points = N_repeat_runs * n_runs * batch_size
    pred_mins = total_points * time_per_point / 60
    print(f"Estimate: {pred_mins:.1f} minutes (~{pred_mins/60:.2f} hours) for {total_points} points")
    
    os.makedirs(outdir, exist_ok=True)
    
    for nth_run in range(N_repeat_runs):
        for j in range(n_runs):
            row = df.iloc[j]
            m_phi_base = float(row["Mh2"])
            m_A_center = float(row["Mh3"])
            m12_center = float(row["m12_2"])
            
            # Make a reproducible seed per run
            seed = base_seed + j
            
            # Generate the local LHC variations
            param_list = generate_local_variations(
                m_phi_base=m_phi_base,
                m_A_center=m_A_center,
                m12_center=m12_center,
                batch_size=batch_size,
                eps_A=eps_A,
                eps_m12=eps_m12,
                seed=seed
            )
            
            # Determine next batch index
            existing = sorted(glob.glob(os.path.join(outdir, "batch_*.pkl")))
            batch_idx = len(existing) + 1
            
            # Run the oracle
            t0 = time.perf_counter()
            results = executor.map(param_list.tolist(), use_threads=True)
            dt = time.perf_counter() - t0
            
            # Save this batch
            filename = f"batch_{batch_idx}_{int(m_phi_base)}.pkl"
            outfile = os.path.join(outdir, filename)
            with open(outfile, "wb") as f:
                pickle.dump({"params": param_list, "results": results}, f)
            
            print(f"[Run {j+1}/{n_runs}] batch {batch_idx} @ m_phi={m_phi_base:.0f} "
                f"saved in {dt:.1f}s → {outfile}")
        print(f"[Epoch Run {nth_run+1}/{N_repeat_runs}]")
    print("All runs completed.")



# Runs

In [28]:
import os
import glob
import pickle
import psutil
import time
import numpy as np
from scipy.stats import qmc


max_merge_size_mb = 30


CSV_PATH   = "valid_points.csv"
OUTDIR     = "data_batches"
N_RUNS     = 2         # how many base points to run
BATCH_SIZE = 3_000       # points per run
EPS_A      = 0.01       # ± variation in m_A
EPS_M12    = 1e-14       # ± variation in m12^2
SEED       = 100


os.makedirs(OUTDIR, exist_ok=True)

existing_batches = sorted(glob.glob(f"{OUTDIR}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
batch_idx

141

In [29]:
multiple_runs(
    csv_path=CSV_PATH,
    N_repeat_runs=N_RUNS,
    batch_size=BATCH_SIZE,
    eps_A=EPS_A,
    eps_m12=EPS_M12,
    outdir=OUTDIR,
    executor=executor
)


Estimate: 63.7 minutes (~1.06 hours) for 138000 points
[Run 1/23] batch 141 @ m_phi=125 saved in 163.9s → data_batches/batch_141_125.pkl
[Run 2/23] batch 142 @ m_phi=125 saved in 101.9s → data_batches/batch_142_125.pkl
[Run 3/23] batch 143 @ m_phi=125 saved in 55.1s → data_batches/batch_143_125.pkl
[Run 4/23] batch 144 @ m_phi=130 saved in 48.8s → data_batches/batch_144_130.pkl
[Run 5/23] batch 145 @ m_phi=140 saved in 48.5s → data_batches/batch_145_139.pkl
[Run 6/23] batch 146 @ m_phi=150 saved in 50.1s → data_batches/batch_146_150.pkl
[Run 7/23] batch 147 @ m_phi=160 saved in 46.6s → data_batches/batch_147_160.pkl
[Run 8/23] batch 148 @ m_phi=170 saved in 55.0s → data_batches/batch_148_170.pkl
[Run 9/23] batch 149 @ m_phi=180 saved in 50.1s → data_batches/batch_149_180.pkl
[Run 10/23] batch 150 @ m_phi=190 saved in 51.5s → data_batches/batch_150_190.pkl
[Run 11/23] batch 151 @ m_phi=200 saved in 53.0s → data_batches/batch_151_200.pkl
[Run 12/23] batch 152 @ m_phi=210 saved in 54.1s →

In [19]:
print(param_list[:5])
print(len(param_list))

print(results[:5])
print(len(results))

# all pickles are made like:
#    pickle.dump({"params": param_list, "results": results}, f)


[[ 2.90972554e+02  2.42671492e+02  9.99906968e-01  1.83445228e+03
   3.58254116e-05  9.63008916e-05  5.10090209e+00]
 [ 3.90106060e+02  3.52647880e+02  9.99982419e-01  9.86387544e+03
   6.51931101e-05  9.44609250e-05  3.14377118e+00]
 [ 1.45053133e+02  3.97831745e+02  9.99910765e-01  3.91033007e+03
  -5.33026162e-05  9.01694304e-05  3.64750044e+00]
 [ 2.24301321e+02  3.20172639e+02  9.99942371e-01  2.13994012e+03
  -7.65944688e-05  1.70070492e-05  2.27629995e+00]
 [ 1.57675251e+02  2.88391895e+02  9.99979244e-01  5.63763949e+03
  -8.95021400e-05 -5.67457529e-05  2.66907545e+00]]
15000
[{'positivity_ok': None, 'unitarity_ok': None, 'perturbativity_ok': None, 'w_h2_bb': None, 'w_h2_tautau': None, 'w_h2_uu': None, 'w_h2_du': None, 'w_h2_ln': None, 'w_h2_vv': [None, None, None], 'w_h2_gaga': None, 'w_h2_Zga': None, 'w_h2_gg': None, 'w_h2_hh': None, 'w_total_h2': None, 'w_total_top': None, 'branching_ratio_h2_gaga': None, 'lambda1': None, 'lambda2': None, 'lambda3': None, 'lambda4': None, '

In [20]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=False)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



5
Batch 5 saved (15000 points) in 863.3s → data_batches/batch_5.pkl


In [ ]:
results

In [ ]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=True)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



1



# Testing Speed

In [ ]:
# Define the sampling sizes
sample_sizes = [1, 10, 100, 1_000, 10_000]

for n in sample_sizes:
    # Generate Latin Hypercube samples in [0,1]^7, then scale
    sampler = qmc.LatinHypercube(d=7)
    sample_unit = sampler.random(n)
    param_list = qmc.scale(sample_unit, param_bounds[:,0], param_bounds[:,1])
    
    # Measure memory before run
    process = psutil.Process()
    mem_before = process.memory_info().rss
    
    # Run and time
    t0 = time.perf_counter()
    results = executor.map(param_list.tolist(), use_threads=False)
    t1 = time.perf_counter()
    
    mem_after = process.memory_info().rss
    delta_mem = (mem_after - mem_before) / (1024**2)  # in MB
    
    # Save raw results for this batch
    with open(f"oracle_results_{n}.pkl", "wb") as f:
        pickle.dump(results, f)
    
    # Record performance
    perf_records.append({
        "n_points": n,
        "time_sec": t1 - t0,
        "mem_delta_MB": delta_mem
    })
    print(f"Completed batch {n}: time={t1-t0:.2f}s, memory Δ={delta_mem:.1f}MB")

# Save performance table
df_perf = pd.DataFrame(perf_records)
df_perf.to_csv("performance_scaling.csv", index=False)



# Merging

In [ ]:
# ------------------------
# Merge old batches if they exceed size threshold
# ------------------------
def merge_batches(folder, batch_prefix="batch_", merged_prefix="merged_", max_size_mb=30):
    # Count existing merged files to avoid overwrite
    existing_merged = sorted(glob.glob(f"{folder}/{merged_prefix}*.pkl"))
    merge_idx = len(existing_merged) + 1
    
    # Only consider raw batch files
    batch_files = sorted(glob.glob(f"{folder}/{batch_prefix}*.pkl"))
    acc_size = 0
    group = []

    for fp in batch_files:
        fsize = os.path.getsize(fp)
        if (acc_size + fsize) / (1024**2) > max_size_mb and group:
            # Merge current group
            merged_data = []
            for gfp in group:
                with open(gfp, "rb") as gf:
                    merged_data.append(pickle.load(gf))
                os.remove(gfp)
            mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
            with open(mout, "wb") as mf:
                pickle.dump(merged_data, mf)
            print(f"Merged {len(group)} batches into {mout}")
            merge_idx += 1
            group, acc_size = [], 0

        group.append(fp)
        acc_size += fsize

    # Merge any remaining files
    if group:
        merged_data = []
        for gfp in group:
            with open(gfp, "rb") as gf:
                merged_data.append(pickle.load(gf))
            os.remove(gfp)
        mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
        with open(mout, "wb") as mf:
            pickle.dump(merged_data, mf)
        print(f"Merged {len(group)} batches into {mout}")
    return merged_data

# Call merge
merged_data = merge_batches(outdir)
merged_data

Merged 2 batches into data_batches/merged_9.pkl


[{'params': array([[ 1.65106260e+02,  1.70883284e+02,  9.63427920e-01,
           2.71242992e+03,  6.53763371e-03, -4.28466529e-03,
           1.43103852e+00],
         [ 3.17185963e+02,  3.09266503e+02,  9.79986220e-01,
           5.52912404e+03, -3.30762866e-03, -2.66594634e-03,
           2.24380465e+00],
         [ 1.39978179e+02,  2.91122058e+02,  9.75932979e-01,
           7.56212901e+03, -1.04069105e-03,  9.94753733e-03,
           1.29141029e-01],
         [ 3.32462488e+02,  1.94804793e+02,  9.59009631e-01,
           4.50933045e+03,  7.85504201e-03, -3.08245371e-04,
           1.24778848e+00],
         [ 2.62605588e+02,  2.52211276e+02,  9.91241966e-01,
           6.49192503e+03, -1.52469795e-03,  1.24466338e-03,
           2.37112012e+00],
         [ 1.45105998e+02,  3.59253506e+02,  9.76235705e-01,
           4.25981235e+03, -5.21246992e-03, -1.92826563e-03,
           1.52882523e+00],
         [ 2.16606118e+02,  4.91637639e+02,  9.72080059e-01,
           5.85320136e+03,  4

In [27]:
merged_data[0]

{'params': array([[ 1.65106260e+02,  1.70883284e+02,  9.63427920e-01,
          2.71242992e+03,  6.53763371e-03, -4.28466529e-03,
          1.43103852e+00],
        [ 3.17185963e+02,  3.09266503e+02,  9.79986220e-01,
          5.52912404e+03, -3.30762866e-03, -2.66594634e-03,
          2.24380465e+00],
        [ 1.39978179e+02,  2.91122058e+02,  9.75932979e-01,
          7.56212901e+03, -1.04069105e-03,  9.94753733e-03,
          1.29141029e-01],
        [ 3.32462488e+02,  1.94804793e+02,  9.59009631e-01,
          4.50933045e+03,  7.85504201e-03, -3.08245371e-04,
          1.24778848e+00],
        [ 2.62605588e+02,  2.52211276e+02,  9.91241966e-01,
          6.49192503e+03, -1.52469795e-03,  1.24466338e-03,
          2.37112012e+00],
        [ 1.45105998e+02,  3.59253506e+02,  9.76235705e-01,
          4.25981235e+03, -5.21246992e-03, -1.92826563e-03,
          1.52882523e+00],
        [ 2.16606118e+02,  4.91637639e+02,  9.72080059e-01,
          5.85320136e+03,  4.09782222e-03, -8.35